In [12]:
from Reporting_Functions import *
import pandas as pd
import numpy as np
from rapidfuzz import process, fuzz

In [13]:
# Load the Excel files

df_1_DATA = load_excel(r"D:\Projects\Lux_Project_Intern\Data\1_DATA.xlsx")
df_2_LIST_OF_COLUMNS = load_excel(r"D:\Projects\Lux_Project_Intern\Data\2_LIST_OF_COLUMNS.xlsx")
df_3_LOB = load_excel(r"D:\Projects\Lux_Project_Intern\Data\3_LOB.xlsx")
df_4_TESTS = load_excel(r"D:\Projects\Lux_Project_Intern\Data\4_TESTS.xlsx")
df_5_FINANCIALS = load_excel(r"D:\Projects\Lux_Project_Intern\Data\5_FINANCIALS.xlsx")

In [14]:
# Clean and align the columns of df_1_DATA based on the column names in df_2_LIST_OF_COLUMNS

df_1_DATA = clean_and_align_columns(df_1_DATA, df_2_LIST_OF_COLUMNS['Column Names'].tolist())

In [15]:
# clean LA_LOB, LA_STATUS columns from df_1_DATA

df_1_DATA = clean_lob_status(
    df_1_DATA,
    lob_col="LA_LOB",
    status_col="LA_STATUS",
    lob_choices=df_3_LOB["Line of business"].dropna().tolist(),
    status_choices=df_3_LOB["Status"].dropna().tolist(),
    threshold=75
)

## #Data tests 

In [16]:
df_1_DATA.head()

,LA_AS_AT_DATE,LA_POLICY_NUMBER,LA_CLAIM_NUMBER,LA_LOB,LA_LOSS_DATE,LA_PAYMENT_DATE,LA_STATUS,LA_GROSS_AMOUNT,LA_CEDED_AMOUNT
0,NaT,22/100/20010/0003589,C/24/100/20/01611,ENGINEERING,2024-05-18,NaT,PAID,734052.79,NaN
1,2025-12-31,23/100/20010/0004000,C/25/100/20/01883,ENGINEERING,2025-05-13,2025-11-20,PAID,806656.00,NaN
2,2025-12-31,21/100/20010/02804,C/25/100/20/01887,ENGINEERING,2025-04-30,2025-12-21,PAID,22000.00,NaN
3,2025-12-31,25/100/20010/0004489,C/25/100/20/01942,ENGINEERING,2025-07-26,2025-12-24,PAID,16500.00,NaN
4,2025-12-31,25/100/20010/0004392,C/25/100/20/01989,ENGINEERING,2025-10-03,2025-12-07,PAID,413750.00,NaN


In [17]:
# test 1- check_missing

print(check_missing(df=df_1_DATA, col_name='LA_LOB'))
print(check_missing(df=df_1_DATA, col_name='LA_LOSS_DATE'))

{'column': 'LA_LOB', 'status': 'PASS', 'message': "No nulls in 'LA_LOB'", 'failed_rows': []}
{'column': 'LA_LOSS_DATE', 'status': 'PASS', 'message': "No nulls in 'LA_LOSS_DATE'", 'failed_rows': []}


In [18]:
# test 2- check_missing

test, df_1_DATA= AS_AT_DATE_CLEAN(df=df_1_DATA, as_at_col='LA_AS_AT_DATE', payment_col='LA_PAYMENT_DATE')

test

{'column': 'LA_AS_AT_DATE',
 'status': 'PARTIALLY_FIXED',
 'message': "4 nulls found in 'LA_AS_AT_DATE' -> 4 rows substituted using quarter-end of 'LA_PAYMENT_DATE'",
 'rows_fixed': 4,
 'remaining_issues': [0, 23, 80]}

In [19]:
# test 3- PAYMENT_DATE_CLEAN

test, df_1_DATA= PAYMENT_DATE_CLEAN(df=df_1_DATA, payment_date_col='LA_PAYMENT_DATE', status_col='LA_STATUS', condition_value='PAID')

test


{'column': 'LA_PAYMENT_DATE',
 'status': 'FAIL',
 'message': "1 dates wiped (status != 'PAID'); 2 PAID rows still missing a payment date",
 'wiped_count': 1,
 'missing_paid_dates': 2}

In [20]:
# test 4- PAYMENT_VS_LOSS_DATE_VALIDATE

test, df_1_DATA= PAYMENT_VS_LOSS_DATE_VALIDATE(df= df_1_DATA, payment_date_col='LA_PAYMENT_DATE', loss_date_col='LA_LOSS_DATE', allow_same_day=True)

test

{'column': 'LA_PAYMENT_DATE vs LA_LOSS_DATE',
 'status': 'PASS',
 'message': '36 rows checked; 47 skipped (missing date(s)); 0 invalid (payment date before loss date)',
 'checked_count': 36,
 'skipped_count': 47,
 'invalid_count': 0,
 'invalid_rows': Empty DataFrame
 Columns: [LA_AS_AT_DATE, LA_POLICY_NUMBER, LA_CLAIM_NUMBER, LA_LOB, LA_LOSS_DATE, LA_PAYMENT_DATE, LA_STATUS, LA_GROSS_AMOUNT, LA_CEDED_AMOUNT, PAYMENT_VS_LOSS_VALIDATION]
 Index: []}

In [21]:
# test 5- VALUATION_VS_LOSS_DATE_VALIDATE (valuation data should be given)

# test, df_1_DATA= VALUATION_VS_LOSS_DATE_VALIDATE()

# test